In [1]:
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server

In [2]:
def get_htmx_idx(hdrs):
    return next((i for i, hdr in enumerate(app.hdrs) if (hdr.attrs.get('src') or '').endswith('htmx.min.js')), -1)

In [3]:
from fasthtml.common import *
import asyncio
from datetime import datetime

In [4]:
app, rt = fast_app()

In [5]:
app.hdrs

[meta((),{'charset': 'utf-8'}),
 meta((),{'name': 'viewport', 'content': 'width=device-width, initial-scale=1, viewport-fit=cover'}),
 script((),{'src': 'https://cdn.jsdelivr.net/npm/htmx.org@2.0.6/dist/htmx.min.js'}),
 script((),{'src': 'https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js'}),
 script((),{'src': 'https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js'}),
 script((),{'src': 'https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js'}),
 (link((),{'rel': 'stylesheet', 'href': 'https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css'}),
  style((':root { --pico-font-size: 100%; }',),{})),
 script(("\n    function sendmsg() {\n        window.parent.postMessage({height: document.documentElement.offsetHeight}, '*');\n    }\n    window.onload = function() {\n        sendmsg();\n        document.body.addEventListener('htmx:afterSettle',    sendmsg);\n        document.body.addEventListener('htmx:wsAfterMessage', sendmsg);\n    };",)

In [6]:
htmx_idx = get_htmx_idx(app.hdrs)
app.hdrs.insert(htmx_idx+1, Script(src="https://unpkg.com/htmx-ext-sse@2.2.3/dist/sse.min.js"))

In [7]:
app.hdrs

[meta((),{'charset': 'utf-8'}),
 meta((),{'name': 'viewport', 'content': 'width=device-width, initial-scale=1, viewport-fit=cover'}),
 script((),{'src': 'https://cdn.jsdelivr.net/npm/htmx.org@2.0.6/dist/htmx.min.js'}),
 script(('',),{'src': 'https://unpkg.com/htmx-ext-sse@2.2.3/dist/sse.min.js'}),
 script((),{'src': 'https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js'}),
 script((),{'src': 'https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js'}),
 script((),{'src': 'https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js'}),
 (link((),{'rel': 'stylesheet', 'href': 'https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css'}),
  style((':root { --pico-font-size: 100%; }',),{})),
 script(("\n    function sendmsg() {\n        window.parent.postMessage({height: document.documentElement.offsetHeight}, '*');\n    }\n    window.onload = function() {\n        sendmsg();\n        document.body.addEventListener('htmx:afterSettle',    sendmsg);\n  

In [8]:
# @rt('/')
# def home():
#     return Div(
#         Div(sse_swap="message", hx_swap="beforeend", id="notifications"),
#         hx_ext="sse", 
#         sse_connect="/events",
#     )

# @rt('/events')
# async def events(req):
#     async def stream():
#         while True:
#             # Send HTML content directly
#             html = Div(f"Update at {datetime.now()}", 
#                       cls="notification")
#             yield f"data: {to_xml(html)}\n\n"
#             await asyncio.sleep(1)
    
#     return EventStream(stream())

In [9]:
import random

@rt
def index(): return Div(hx_ext="sse", sse_connect="/numstream", hx_swap="beforeend show:bottom", sse_swap="message")

# `signal_shutdown()` gets an event that is set on shutdown
shutdown_event = signal_shutdown()

async def number_generator():
    while not shutdown_event.is_set():
        data = Article(random.randint(1, 100))
        yield sse_message(data)
        await asyncio.sleep(1)

@rt
async def numstream(): return EventStream(number_generator())

In [11]:
signal_shutdown??

Signature: signal_shutdown()
Docstring: <no docstring>
Source:   
def signal_shutdown():
    from uvicorn.main import Server
    event = asyncio.Event()
    @patch
    def handle_exit(self:Server, *args, **kwargs):
        event.set()
        self.force_exit = True
        self._orig_handle_exit(*args, **kwargs)
    return event
File:      ~/miniforge3/envs/cjm-tqdm-capture/lib/python3.11/site-packages/fasthtml/core.py
Type:      function

In [10]:
# Start server for Jupyter
server = start_test_server(app)
display(HTMX())

In [11]:
# Stop server when done
server.stop()